# **Aula : Problemas de Interesse**

## Introdução

## **Allen-Cahn**

### **Equação Governante** 

A equação governante adotada segue a forma:
  $$\frac{\partial u}{\partial t} = \mu \left( \epsilon^2 \frac{\partial^2 u}{\partial x^2} - f(u) \right)$$

---

### **Fluxo Real**
Existem duas configurações comuns para o contexto de Allen-Cahn.

Na primeira o termo não-linear $f(u)$ no código é parametrizado como:
$$f(u) = -\beta_{fe} u$$

Na segunda o potencial duplo (cúbico), que é o mais tradicional para o problema, é definido por:
$$f(u) = u^3 - u$$

---

### **Condições Iniciais (IC)** 

* **Analítica:** O sistema é inicializado no domínio $x \in [0, 4\pi]$ com um perfil cossenoidal:
$$u(x,0) = \cos(0.5x)$$

* **Periódica:** O estado inicial no domínio $x \in [0, 2\pi]$ é uma onda senoidal transladada verticalmente:
$$u(x,0) = 0.8 + \sin(x)$$

---

### **Condições de Contorno (BC)** 

No contexto do DG a condição de contorno pode ser feita via *Ghost Cells*, impondo a forma fraca por meio do fluxo numérico na interface da borda. Para tal fluxo, calcula-se o estado numérico do traço (a média entre o elemento interno $u_{int}$ e o estado fantasma externo $u_{ghost}$):

$$u_{borda}^* = \frac{u_{int} + u_{ghost}}{2}$$

Se queremos que o valor físico efetivamente aplicado na borda seja o valor prescrito $g(t)$, igualamos o traço a $g(t)$:
$$\frac{u_{int} + u_{ghost}}{2} = g(t) \implies \boxed{u_{ghost} = -u_{int} + 2g(t)}$$

```Python
#Recovers the physical solution uh at the left boundary of the first element and at
#the right boundary of the last element (using the first row of matrices Flkp1 and Frk)
uhL = np.dot(Flkp1[0,:],ut[:,1,0])
uhR = np.dot(Frk[0,:],ut[:,-2,0])
```

* `uhL` representa $u_{int}$ (o valor da solução interpolado na borda interna esquerda).
* `uhR` representa $u_{int}$ (o valor da solução interpolado na borda interna direita).

#### **Análitica (Dirichlet Não Homogêneo)**
* Física: É aplicada uma condição de Dirichlet dependente do tempo baseada na evolução da solução analítica exata nas bordas:

$$u(0,t) = e^{-t} \cos(0.5x_{min})$$
$$u(L,t) = e^{-t} \cos(0.5x_{max})$$

```Python
# Analytical (Non Homogeneous Dirichlet)
ut[0,0,0] = -uhL + 2.0*(np.exp(-t)*np.cos(0.5*xmin)) #entrance
ut[0,-1,0] = -uhR + 2.0*(np.exp(-t)*np.cos(0.5*xmax)) #exit
```

#### **Periódica (Dirichlet Não Homogêneo)** 
* Física: Fixa os valores das bordas no tempo $t$ com os mesmos valores da Condição Inicial em $t=0$.
* Contexto: Usado quando se quer simular um domínio onde os reservatórios nas extremidades são mantidos com concentrações/fases constantes.

```Python
# Non homogeneous Dirichlet
ut[0,0,0] = -uhL + 2.0*(0.8 + np.sin(xc[0,0]))    #entrance (Dirichlet)
ut[0,-1,0] = -uhR + 2.0*(0.8 + np.sin(xc[0,0]))    #exit (Dirichlet)
```

#### **Dirichlet Homogêneo**
* Física: $u(0,t) = 0$ e $u(L,t) = 0$.
* Contexto: Representa o estado de fase neutro $u=0$ nas paredes/fronteiras.

```Python
# Homogeneous Dirichlet
ut[0,0,0] = -uhL    #entrance (Dirichlet)
ut[0,-1,0] = -uhR    #exit (Dirichlet)
```

#### **Neumann Homogêneo**

* Física: o estado adjacente da borda copia o estado do elemento:

$$\left. \frac{\partial u}{\partial x} \right |_{x=0} = 0 \quad \text{e} \quad \left. \frac{\partial u}{\partial x} \right |_{x=L} = 0$$

* Contexto: Como $u_{ghost} = u_{int}$, a diferença na interface é nula ($[[u]] = 0$). Esta é a condição de contorno mais clássica e fisicamente natural para a Equação de Allen-Cahn, pois representa um sistema isolado (fronteiras impermeáveis), onde a dinâmica de separação de fases ocorre sem troca de massa com o meio externo. 

```Python
# Homogeneous Neumann at left and right boundaries
ut[0,0,0] = uhL     #entrance 
ut[0,-1,0] = uhR    #exit
```

#### **Periódica**
* Física: Simula um domínio infinito ou "em anel", onde o fluxo que sai pela direita entra pela esquerda ($u(0,t) = u(L,t)$).
* Contexto: Muito útil em Allen-Cahn para estudar o crescimento e aniquilação de domínios (interfaces de fase) longe de qualquer efeito de parede/fronteira. 

```Python
# Periodic BC
ut[0,0,0] = -uhL + 2.0*(0.8 + np.sin(xc[0,0])) #entrance
ut[0,-1,0] = ut[0,0,0]  #exit
```

---
 

## **Cahn-Hillard**

### **Equação Governante**

Aqui temos um sistema acoplado Cahn-Hilliard / Allen-Cahn, lidando com dois parâmetros de ordem: $u$ (fase conservada, regida por Cahn-Hilliard) e $v$ (fase não-conservada, regida por Allen-Cahn).  As equações governantes adotadas seguem a a forma:

$$\frac{\partial u}{\partial t} = \frac{\partial}{\partial x} \left[ b(u,v) \frac{\partial}{\partial x} \left( \frac{\partial \Psi}{\partial u} - \gamma \frac{\partial^2 u}{\partial x^2} \right) \right]$$

$$\frac{\partial v}{\partial t} = - \rho^{-1} b(u,v) \left( \frac{\partial \Psi}{\partial v} - \gamma \frac{\partial^2 v}{\partial x^2} \right)$$

---

### **Fluxo Real**
O fluxo e as não-linearidades deste sistema são divididos em duas partes: a função de mobilidade $b(u,v)$ e as derivadas da Energia Livre $\Psi(u,v)$. 

#### Função de Mobilidade $b(u,v)$:
Na primeira configuração (geralmente usada com condições flutuantes), a mobilidade é dependente do estado:
$$b(u,v) = u(1-u)(0.25 - v^2)$$

Na segunda configuração (usada para validação analítica), a mobilidade é constante:
$$b(u,v) = 1.0$$

#### Derivadas da Energia Livre $\Psi(u,v)$:

A não-linearidade termodinâmica inclui interações entrópicas (termos logarítmicos) e entálpicas:

$$\frac{\partial \Psi}{\partial u} = \Theta \left[ \ln\left(\frac{u+v}{1-u-v}\right) + \ln\left(\frac{u-v}{1-u+v}\right) \right] + \frac{\alpha}{2}(1-2u)$$

$$\frac{\partial \Psi}{\partial v} = \Theta \left[ \ln\left(\frac{u+v}{1-u-v}\right) - \ln\left(\frac{u-v}{1-u+v}\right) \right] - \beta v$$

Uma versão simplificada com o termo $\Theta$ desligado para testes puramente analíticos também pode ser usada:

$$\frac{\partial \Psi}{\partial u} = \frac{\alpha}{2}(1-2u)$$

$$\frac{\partial \Psi}{\partial v} = - \beta v$$

---

### **Condições Iniciais**
* **Analítica:** O sistema é inicializado para evoluir de acordo com a solução exata esperada:
$$u(x,0) = \cos(0.5x) - \sin(x)$$
$$v(x,0) = \cos(0.5x)$$

* **Manufaturada:** Outra configuração suave para testes de convergência:
$$u(x,0) = 0.5 - e^{-2.0}\sin(x)$$
$$v(x,0) = e^{-4.0}\cos(0.5x)$$

* **Flutuante (Determinística):** Uma perturbação senoidal com decaimento exponencial, usada para iniciar a separação de fases:
    * Para $u(x,0)$ existem três configurações possíveis
            $$u(x,0) = 0.55 - e^{-3.0}\sin(6\pi x)$$
            $$u(x,0) = 0.55 - e^{-3.0}\sin(6\pi x + \frac{10}{21}\pi)$$
            $$u(x,0) = 0.55 - e^{-3.0}\sin(6\pi x - \frac{10}{21}\pi)$$
    * Para $v(x,0)$ existem duas configurações possíveis   
            $$v(x,0) = e^{-4.0}\cos(2\pi x)$$
            $$v(x,0) = 0.01$$            

* **Flutuante (Estocástica / Arquivo):** Inserção de ruído aleatório em $u$ e/ou $v$ e a leitura de um perfil prévio para $u$ e/ou $v$:
  * Ruído Aleatório:
    $$u(x,0) = \text{Uniforme}(-0.05, 0.05)$$
    $$v(x,0) = \text{Uniforme}(-0.002, 0.002)$$
  * Com arquivo prévio
    $$u(x,0) = \text{Carregado de arquivo (icr\_u.npy)}$$
    $$v(x,0) = \text{Carregado de arquivo (icr\_v.npy)}$$


---

### **Condições de Contorno**
Assim como explicado no Allen-Cahn se queremos que o valor físico efetivamente aplicado na borda seja o valor prescrito $g(t)$
$$u_{ghost} = -u_{int} + 2g(t)$$

```Python
#Recovers the physical solution uh at the left boundary of the first element and at
#the right boundary of the last element (using the first row of matrices Flkp1 and Frk)
uhR = np.dot(Frk[0,:],ut[:,-2,0])
vhR = np.dot(Frk[0,:],vt[:,-2,0])
uhL = np.dot(Flkp1[0,:],ut[:,1,0])
vhL = np.dot(Flkp1[0,:],vt[:,1,0])
```

#### **Analítica (Dirichlet Não Homogêneo)**
* Física: É aplicada uma condição de Dirichlet dependente do tempo, baseada na evolução temporal forçada nas bordas para ambas as variáveis.  
* Contexto: Usado com o Método das Soluções Manufaturadas (MMS) para validar a precisão e a estabilidade do solver.

```Python
# Non homogeneous Dirichlet
ut[0,-1,0] = -uhR + 2.0*(np.exp(t)*np.cos(0.5*xmax) - np.exp(-0.5*t)*np.sin(xmax)) #exit
vt[0,-1,0] = -vhR + 2.0*(np.exp(-t)*np.cos(0.5*xmax)) #exit
ut[0,0,0] = -uhL + 2.0*(np.exp(t)*np.cos(0.5*xmin) - np.exp(-0.5*t)*np.sin(xmin)) #entrance
vt[0,0,0] = -vhL + 2.0*(np.exp(-t)*np.cos(0.5*xmin)) #entrance
```

#### **Dirichlet Homogêneo**
* Física: Fixa o valor das variáveis em zero nas extremidades do domínio ($u=0, v=0$).  
* Contexto: Representa o ancoramento de uma fase específica (fase neutra) nas paredes do sistema físico.

``` Python
# Homogeneous Dirichlet
ut[0,0,0] = -uhL    #entrance (Dirichlet)
ut[0,-1,0] = -uhR   #exit (Dirichlet)
```

#### **Neumann Homogêneo**
* Física: O estado adjacente da borda copia o estado do elemento, forçando gradientes nulos nas fronteiras para ambos os parâmetros:
$$\left. \frac{\partial u}{\partial x} \right |_{borda} = 0 \quad \text{e} \quad \left. \frac{\partial v}{\partial x} \right |_{borda} = 0$$

* Contexto: Esta é a condição física mais importante para Cahn-Hilliard, pois impõe fluxo de massa nulo. Fronteiras impermeáveis garantem que a massa total do parâmetro conservado $u$ permaneça constante durante a separação de fases.

```Python
# Homogeneous Neumann at left and right boundaries
ut[0,-1,0] = uhR
vt[0,-1,0] = vhR
```

> Nota 1: O mesmo tratamento é matematicamente replicado ou estendido para a borda esquerda e para as variáveis auxiliares do sistema acoplado, como os fluxos $q_1$ e $r_1$

> Nota 2: Existe o uso de **duas projeções de fluxo** no Cahn-Hillard, a *MobilityProjection* e a *FreeEnergyProjection*

---

## **Euler**

### **Equação Governante**

O problema físico é o das Equações de Euler para a Dinâmica de Gases em 1D. Trata-se de um sistema hiperbólico de leis de conservação que modela o escoamento de um fluido invíscido (sem viscosidade física) e compressível.  A equação governante adotada é expressa na sua forma vetorial:

$$\frac{\partial \mathbf{u}}{\partial t} + \frac{\partial \mathbf{f}(\mathbf{u})}{\partial x} = 0$$

onde o vetor de variáveis conservadas $\mathbf{u}$ abrange a densidade ($\rho$), o momento ($\rho u$) e a energia total ($E$):
$$\mathbf{u} = \begin{bmatrix} \rho \\ \rho u \\ E \end{bmatrix}$$

---

### **Fluxo Real**

O vetor de fluxo físico $\mathbf{f}(\mathbf{u})$ para as equações de Euler unidimensionais, representa o transporte das quantidades conservadas e é definido como:

$$\mathbf{f}(\mathbf{u}) = \begin{bmatrix} \rho u \\ \dfrac{(\rho u)^2}{\rho} + p \\ (E + p)\dfrac{\rho u}{\rho} \end{bmatrix}$$

A pressão $p$ acopla o sistema termodinamicamente usando a equação de estado para um gás ideal:

$$p = (\gamma - 1)\left( E - \dfrac{1}{2}\frac{(\rho u)^2}{\rho} \right)$$

onde $\gamma$ (razão de calores específicos) é **fixado em 1.4**, o valor típico para o ar.

---

### **Condições Iniciais**

**Tubo de Choque de Sod (*Sod Shock Tube*)**: O sistema modela um tubo fechado particionado ao meio, separando um gás em alta pressão/densidade de um gás em baixa pressão/densidade, ambos inicialmente em repouso. O domínio é $x \in [0, 1]$ com o diafragma localizado no centro ($x = 0.5$). 

As variáveis são inicializadas da seguinte forma:
$$\rho(x,0) = \begin{cases} 1.0, & x \le 0.5 \\ 0.125, & x > 0.5 \end{cases}$$

$$\rho u(x,0) = 0.0$$

$$E(x,0) = \begin{cases} \dfrac{1.0}{\gamma - 1}, & x \le 0.5 \\ \dfrac{0.1}{\gamma - 1}, & x > 0.5 \end{cases}$$

---

### **Condições de Contorno**

Assim como visto anteriormente, o método DG impõe as condições de contorno de forma fraca utilizando a técnica de Ghost Cells ($u_{ghost} = -u_{int} + 2g(t)$) para impor os valores exatos de contorno.  Neste problema, o vetor de estado tem 3 componentes, logo, a técnica é aplicada para a densidade, o momento e a energia.

#### **Dirichlet Não Homogêneo (Estático)**
* Física: Fixa os estados termo-fluidinâmicos nas bordas esquerda e direita com os mesmos valores exatos dos estados da Condição Inicial (alta pressão à esquerda, baixa pressão à direita).  
* Contexto: Ao simular um tubo de choque fechado por um período curto ($t=0.2$ s), as ondas (choque, rarefação e descontinuidade de contato) não têm tempo hábil para atingir e refletir nas paredes. Portanto, as extremidades do domínio comportam-se como reservatórios infinitos que conservam seus estados originais intocados.

```Python
# Non homogeneous Dirichlet at left/right boundaries
rhoin = 1.0
rhouin = 0.0
pin = 1.0
Enerin = pin/(gamma-1.0)

rhoout = 0.125
rhouout = 0.0
pout = 0.1
Enerout = pout/(gamma-1.0)        

# Aplicação da BC nas Ghost Cells (Entrance = Borda Esquerda, Exit = Borda Direita)
rhot[0,0,0] = -rhohL + 2.0*rhoin    #entrance
rhot[0,-1,0] = -rhohR + 2.0*rhoout  #exit
   
rhout[0,0,0] = -rhouhL + 2.0*rhouin    #entrance
rhout[0,-1,0] = -rhouhR + 2.0*rhouout  #exit

pint = -pres[0,0] + 2.0*pin   #entrance
poutt = -pres[-1,-1] + 2.0*pout   #exit

Enert[0,0,0] = -EnerhL + 2.0*Enerin    #entrance
Enert[0,-1,0] = -EnerhR + 2.0*Enerout  #exit
```

> Nota para implementação: A pressão na interface também é projetada fracamente por `pint` e `poutt` para posteriormente ser injetada no cálculo explícito das variáveis de fluxo na borda.

---

## **Burgers Inviscido**

### **Equação Governante**
O problema físico é a clássica Equação de Burgers Invíscida (Inviscid Burgers' equation) em 1D. É uma equação diferencial parcial não-linear fundamental para o estudo de fluidos, ondas de choque e leis de conservação hiperbólicas. 
A equação governante adotada segue a forma:
$$\frac{\partial u}{\partial t} + \frac{\partial f(u)}{\partial x} = 0$$

---

### **Fluxo Real**
Diferente dos problemas anteriores, a Equação de Burgers possui apenas uma única configuração padrão para a função de fluxo real. O fluxo não-linear é parametrizado estritamente como a energia cinética convectiva:
$$f(u) = \frac{u^2}{2}$$

---

### **Condições Iniciais**
Existe uma gama muito rica de condições iniciais para testar o comportamento de esquemas numéricos (Viscosidade Artificial) frente a descontinuidades e choques, assim como testar os limitadores de inclinação (Slope Limiters) usando o caso principal. As configurações de $u(x,0)$ mapeadas são:

* Inviscid Burgers (Onda Senoidal Principal): O clássico problema de desenvolvimento de choque a partir de uma onda harmônica suave no domínio $x \in [0, 2\pi]$:
$$u(x,0) = 0.5 + \sin(2\pi x)$$

* Suave (Smooth): Uma meia-onda senoidal:
$$u(x,0) = \sin(0.5\pi x)$$

* Onda Quadrada (Square Wave): Inicialização em degrau com patamar superior entre 0.5 e 2.0:
$$u(x,0) = \begin{cases} 1.0, & 0.5 \le x \le 2.0 \\ 0.0, & \text{caso contrário} \end{cases}$$

* Choque Central (Center Shock): Uma descontinuidade centralizada exata:
$$u(x,0) = \begin{cases} 1.0, & x \le 2.0 \\ -1.0, & x > 2.0 \end{cases}$$

---

### **Condições de Contorno**

A mecânica de Ghost Cells é mantida para impor as condições de contorno em sua formulação fraca, onde o estado numérico externo é forçado como $u_{ghost} = -u_{int} + 2g(t)$.

```Python
#Recovers the physical solution uh at the left boundary of the first element and at
#the right boundary of the last element (using the first row of matrices Flkp1 and Frk)
uhL = np.dot(Flkp1[0,:],ut[:,1,0])
uhR = np.dot(Frk[0,:],ut[:,-2,0])
```

#### **Dirichlet Não Homogêneo (Dependente do Tempo)**
* Física: Aplica uma perturbação baseada em uma onda senoidal (ou valor exato esperado analiticamente) com base no tempo e na velocidade de face $u_L$.
* Contexto: Utilizada na simulação principal da equação de Burgers para forçar o escoamento a acompanhar o desenvolvimento do choque de uma onda senoidal. O código copia a mesma condição da entrada para a saída.

```Python
#Inviscid Burgers' example
ut[0,0,0] = -uhL + 2.0*(0.5 + np.sin(-2.0*np.pi*uhL*t)) #entrance
ut[0,-1,0] = ut[0,0,0]  #exit
```

#### **Dirichlet Homogêneo**
* Física: $u(0,t) = 0$ e $u(L,t) = 0$.
* Contexto: Usado para simulações puras de translação e decaimento (como a configuração de Smooth e Square Wave), onde o valor natural nas fronteiras é o repouso do sistema.

```Python
#Smooth and Square wave IC examples
ut[0,0,0] = -uhL    #entrance
ut[0,-1,0] = -uhR   #exit
```

#### **Dirichlet Não Homogêneo (Estático / Choque Central)**
* Física: Força $u(0,t) = 1.0$ e $u(L,t) = -1.0$ nas bordas.
* Contexto: Aplicada em conjunto com o caso do "Choque Central". Mantendo a esquerda injetando massa a uma velocidade de $+1$ e a direita a uma velocidade de $-1$, as ondas se chocam exatemente no centro do domínio e o perfil do choque estacionário deve permanecer perfeitamente centralizado e estático ao longo do tempo.

```Python
#Center shock example
ut[0,0,0] = -uhL + 2.0*(1.0)      #entrance
ut[0,-1,0] = -uhR + 2.0*(-1.0)    #exit
```

---

## **Advecção Linear**

### **Equação Governante**

Trata-se da lei de conservação hiperbólica mais elementar, que modela o transporte puro de uma grandeza escalar em um campo de velocidade.  A EDP matemática principal que rege o problema é:
$$\frac{\partial u}{\partial t} + \frac{\partial f(u)}{\partial x} = 0$$

---

### **Fluxo Real**
Para o caso da advecção linear, a função de fluxo físico é estritamente linear e depende de uma velocidade de onda de propagação $c$. 
$$f(u) = c u$$

> Para o contexto atual o valor pode ser fixado em $c = 0.5$ 

---

### **Condições Iniciais**
* Onda Quadrada / Advecção Linear: Um perfil de pulso retangular ou "cartola", focado em avaliar como o método lida com descontinuidades fortes durante o transporte no domínio $x \in [0.0, 4.0]$. A função é estruturada como:
$$u(x,0) = \begin{cases} 1.0, & 1.0 \le x \le 3.0 \\ 0.0, & \text{caso contrário} \end{cases}$$

* Suave: Uma onda harmônica básica sem gradientes infinitos:
$$u(x,0) = \sin(x)$$

---

### **Condições de Contorno**
O tratamento numérico obedece à mesma regra de fluxos e Ghost Cells estabelecida nas simulações anteriores para imposição da forma fraca na fronteira ($u_{ghost} = -u_{int} + 2g(t)$).

```Python
#Recovers the physical solution uh at the left boundary of the first element and at
#the right boundary of the last element (using the first row of matrices Flkp1 and Frk)
uhL = np.dot(Flkp1[0,:],ut[:,1,0])
uhR = np.dot(Frk[0,:],ut[:,-2,0])
```

#### **Dirichlet Homogêneo** 
* Física: Força o estado na extremidade do domínio a ser estritamente zero ($u=0$) o tempo todo.
* Contexto: Utilizada ativamente no script para garantir que os contornos externos atuem como zonas de vazio. Como a onda quadrada começa e termina dentro do domínio, essa BC assegura que nenhum ruído numérico entre pelas bordas enquanto o pulso é transportado, evidenciando puramente o comportamento advectivo e a atuação do filtro de captura de choque.

```Python
#Linear Advection example: homogeneous Dirichlet uL = uR = 0
ut[0,0,0] = -uhL  #entrance
ut[0,-1,0] = -uhR  #exit
```

#### **Dirichlet Não Homogêneo** 
* Física: Forçaria uma entrada dependente do tempo com base em uma onda senoidal ou fixa.
* Contexto: Seria útil caso o teste exigisse o acompanhamento analítico da onda transpassando as fronteiras do domínio.

```Python
#Linear Advection example: homogeneous Dirichlet uL = uR = 0
ut[0,0,0] = -uhL - 2.0*np.sin(2.0*np.pi*t)
ut[0,0,0] = -uhL + 2.0*(1.0)  #entrance
```

#### **Neumann Homogêneo (Comentada)** 
* Física: Força o gradiente espacial da propriedade na borda ser zero ($\frac{\partial u}{\partial x} = 0$).
* Contexto: Extremamente utilizada em problemas de advecção pura no contorno de saída (borda direita, neste caso onde $c>0$). Essa BC (também chamada de condição de contorno outflow) permite que a onda saia do domínio de forma suave sem gerar reflexões.

```Python
#Homogeneous Neumann at right boundary
ut[:,-1,0] = ut[:,-2,0].copy()
ut[0,-1,0] = uhR
```

---

## **Shallow-Water**

### **Equação Governante**

Diferente da formulação clássica baseada em conservação pura de massa e momento ($h$ e $hu$), apresentaremos as Equações de Águas Rasas 1D em variáveis não-conservativas (ou primitivas), onde o vetor de estado evoluído é composto pela altura da lâmina d'água ($h$) e pela velocidade do escoamento ($u$).

As equações diferenciais que regem o sistema acoplado são apresentadas na forma:
$$\frac{\partial h}{\partial t} + \frac{\partial f_h(h,u)}{\partial x} = 0$$
$$\frac{\partial u}{\partial t} + \frac{\partial f_u(h,u)}{\partial x} = S_u(h,u)$$

---

### **Fluxo Real e Termo Fonte**
O problema é subdividido em escoamentos sobre canais horizontais (sem termo fonte) e canais inclinados com atrito de fundo (com termo fonte e decomposição da gravidade).

#### **Vetor de Fluxo Não-Linear:**
O fluxo advectivo para as variáveis não-conservativas é modelado como:
$$f_h(h,u) = h u$$
$$f_u(h,u) = g \cos(\alpha) h + \frac{u^2}{2}$$

Onde $\alpha$ é o ângulo de inclinação do leito. Quando focado no caso horizontal plano, o fluxo simplifica para 
$$f_u = gh + u^2/2$$

#### **Termo Fonte ($S_u$):**
Ao incluir forças externas de gravidade (aceleração devido à rampa) e dissipação por atrito de fundo:
$$S_u(h,u) = g \sin(\alpha) - C_f \frac{u^2}{2h}$$

Onde o coeficiente de atrito $C_f$ é calculado dinamicamente com base no Número de Reynolds (Regime Laminar usando a **equação de Blasius**, e Regime Turbulento usando a **equação de Churchill**, lidando com a rugosidade $\epsilon$).

O cálculo do número de Reynolds nessa ocasião é dado por:
$$Re = \dfrac{4h u}{\nu}$$
em que $\nu$ é a viscoidade (por padrão $\nu = 1 \times 10^{-6}$)

$$
C_f = \begin{cases}
\dfrac{16}{Re}  & \quad \text{Se} \ Re \leq 2000 & \leftarrow & \text{Blasius}\\
2 \left[ \left(\dfrac{8.0}{Re}\right)^{12} + (A + B)^{-1.5} \right]^{1/12}  & \quad \text{Caso contrário} & \leftarrow & \text{Churchill}
\end{cases}
$$

sendo as seguintes definições para Churchill
$$
D = 4h \hspace{3cm} B = \Big(\dfrac{37530}{Re} \Big)^{16}
\hspace{3cm}
A = \Big (2.457 \log \Big \{ \Big [\dfrac{7}{Re} \Big ]^{0.9} + 0.27\Big [\dfrac{\epsilon}{D} \Big ]^{-1} \Big\} \Big)^{16}
$$

com $\epsilon = 1\times10^{-3}$.

---

### **Condições Iniciais**

* Solução Analítica Exata (`ShallowIC`): Inicialização de um perfil parabólico em $h$ projetado para evoluir junto a uma solução analítica conhecida em problemas testes de translação:
$$h(x,0) = x^2$$
$$u(x,0) = 2\sqrt{g}(x - \sqrt{H})$$

* Onda de Perturbação Gaussiana (`GaussianIC`): Uma perturbação no formato de um sino de Gauss somada à altura nominal ($H_0$), acompanhada do cálculo da velocidade base para o estado estacionário ($u_0$):
$$h(x,0) = H_0 \left[ A \exp\left(-\frac{(x-\mu)^2}{\sigma^2}\right) + 1 \right]$$
$$u(x,0) = u_0$$

* Ressalto Móvel (`MobileJumpIC`): Uma clássica condição inicial de "quebra de barragem" (Dam Break), que cria uma descontinuidade na elevação da água da metade esquerda para a metade direita do domínio:
$$h(x,0) = \begin{cases} 1.05 H_0 \text{ (ou } 1.1 H_0 \text{)}, & x \le L/2 \\ H_0, & x > L/2 \end{cases}$$

---

### **Condições de Contorno**
O tratamento usa Ghost Cells, mas há uma nuance interessante na implementação destes códigos em comparação aos problemas anteriores. Eles transitam entre aplicar a imposição fraca clássica ($u_{ghost} = -u_{int} + 2g(t)$) e, para alguns casos, atribuir diretamente o valor da variável de fronteira à célula fantasma ($u_{ghost} = g(t)$).

#### **Dirichlet Não Homogêneo (Estados Fixos de Vazão)**
* Física: Força a manutenção de uma altura e velocidade específicas na entrada (upstream) e saída (downstream) do canal.  
* Contextos: 
    * Para o caso da simulação com **Ressalto Móvel**, em que se assegura que o sistema continue sendo alimentado com massa de água à esquerda e escoando à direita.   
        ```Python
        hin = 1.1*H0
        hout = H0    
        uin = (1/1.5)*U0
        uout = U0
        ```
   * Para o caso de uma **perturbação Gaussiana**:  
        ```Python
        hin = H0
        hout = H0    
        uin = U0
        uout = U0
        ```

Em ambas as situações impõe-se a Ghost Cell de forma similar
```Python
# Imposição fraca nas Ghost Cells
ht[0,0,0] = -hhL + 2.0*hin    #entrance
ht[0,-1,0] = -hhR + 2.0*hout  #exit
ut[0,0,0] = -uhL + 2.0*uin    #entrance
ut[0,-1,0] = -uhR + 2.0*uout  #exit
```

#### **Dirichlet Homogêneo para Velocidade (Parede Refletiva)**
* Física: Impõe que a velocidade nas extremidades do canal seja nula ($u=0$).
* Contexto: Utilizada no teste puramente numérico para criar uma fronteira de tipo "parede fechada", forçando as ondas a rebaterem e não saírem do domínio.

```Python
ut[0,0,0] = -uhL    #entrance (resultará na média u_borda = 0)
ut[0,-1,0] = -uhR   #exit (resultará na média u_borda = 0)
```

#### **Dirichlet Analítico Dependente do Tempo** 
* Física: Forçaria $h(x,t)$ e $u(x,t)$ a assumirem, a cada instante $t$, os valores analíticos baseados nas características das ondas:
$$\xi(x,t) = \frac{x + 2\sqrt{gH}t}{1 + 3\sqrt{g}t}$$
$$h_{borda}(t) = \xi(x_{borda},t)^2$$
$$u_{borda}(t) = 2\sqrt{g}\xi(x_{borda},t) - 2\sqrt{gH}$$

* Contexto: Essa BC é usada para validação precisa da convergência do esquema numérico com as Soluções Exatas (MMS - Method of Manufactured Solutions).

```Python
#Non homogeneous Dirichlet at left/right boundaries
hin = ((0 + 2.0*np.sqrt(g*H)*t) / (1.0 + 3.0*np.sqrt(g)*t))**2
uin = 2.0*np.sqrt(g)*((2.0*np.sqrt(g*H)*t) / (1.0 + 3.0*np.sqrt(g)*t)) - 2.0*np.sqrt(g*H)

hout = ((L + 2.0*np.sqrt(g*H)*t) / (1.0 + 3.0*np.sqrt(g)*t))**2
uout = 2.0*np.sqrt(g)*((L + 2.0*np.sqrt(g*H)*t) / (1.0 + 3.0*np.sqrt(g)*t)) - 2.0*np.sqrt(g*H)    

```

---

## **Shock-Density**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Shu-Osher**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Traffic-Flow**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**

## **Burgers Viscoso**

### **Equação Governante**

### **Fluxo Real**

### **Condições Iniciais**

### **Condições de Contorno**